In [105]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
# sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set path to datasets
DATA_DIR = Path("/home/tella/code/StellaRodriguesLallement/OSE_Project/Dataset_visualization/src/ose_core/data_ingestion/extracted_datasets")

# Check it exists
print(DATA_DIR.exists())

print("Libraries imported successfully!")

True
Libraries imported successfully!


In [106]:
# LOADING DATASETS

df_basic     = pd.read_csv(DATA_DIR / '01_company_basic_info.csv', dtype={'siren': str})
df_financial = pd.read_csv(DATA_DIR / '02_financial_data.csv', dtype={'siren': str})
df_workforce = pd.read_csv(DATA_DIR / '03_workforce_data.csv', dtype={'siren': str})
df_structure = pd.read_csv(DATA_DIR / '04_company_structure.csv', dtype={'siren': str})
df_flags     = pd.read_csv(DATA_DIR / '05_classification_flags.csv', dtype={'siren': str})
df_contacts  = pd.read_csv(DATA_DIR / '06_contact_metrics.csv', dtype={'siren': str})  # to be verified
df_kpi       = pd.read_csv(DATA_DIR / '07_kpi_data.csv', dtype={'siren': str})
df_signals   = pd.read_csv(DATA_DIR / '08_signals.csv', dtype={'siren': str})
df_articles  = pd.read_csv(DATA_DIR / '09_articles.csv', dtype={'siren': str})         # not used

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [107]:
df_basic.columns

Index(['company_name', 'siren', 'siret', 'departement', 'resume_activite',
       'raison_sociale_keyword', 'raison_sociale', 'last_modified',
       'processedAt', 'updatedAt'],
      dtype='object')

In [108]:
df_basic.shape

(375, 10)

In [109]:
# BASIC FEATURES
df_basic_feat = df_basic[['company_name','siren', 'siret', 'departement']].copy()
df_basic_feat.head()

,company_name,siren,siret,departement
0,PROCONI,000132066,NaN,00
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21
2,JULIEN MACK,016450298,1.645030e+12,92
3,OROC BAT,046580031,4.658003e+12,64
4,MINOTERIE DU TRIEVES CORREARD ET FILS,057504649,5.750465e+12,38


In [110]:
df_financial.columns.tolist()

['company_name',
 'siren',
 'siret',
 'caConsolide',
 'caGroupe',
 'resultatExploitation',
 'dateConsolide',
 'kpi_2025_capital_social',
 'kpi_2025_evolution_ca',
 'kpi_2023_ca_france',
 'kpi_2023_ca_bilan',
 'kpi_2023_resultat_exploitation',
 'kpi_2023_capital_social',
 'kpi_2023_resultat_bilan',
 'kpi_2022_ca_france',
 'kpi_2022_ca_bilan',
 'kpi_2022_resultat_exploitation',
 'kpi_2022_capital_social',
 'kpi_2022_resultat_bilan',
 'kpi_2021_ca_france',
 'kpi_2021_ca_bilan',
 'kpi_2021_resultat_exploitation',
 'kpi_2021_capital_social',
 'kpi_2021_resultat_bilan',
 'kpi_2020_ca_france',
 'kpi_2020_ca_bilan',
 'kpi_2020_resultat_exploitation',
 'kpi_2020_capital_social',
 'kpi_2020_resultat_bilan',
 'kpi_2019_ca_france',
 'kpi_2019_ca_bilan',
 'kpi_2019_resultat_exploitation',
 'kpi_2019_capital_social',
 'kpi_2019_resultat_bilan',
 'kpi_2018_ca_bilan',
 'kpi_2018_resultat_exploitation',
 'kpi_2018_capital_social',
 'kpi_2018_resultat_bilan',
 'kpi_2017_ca_france',
 'kpi_2017_ca_bilan',

In [111]:
df_financial.shape

(375, 112)

In [112]:
import re

# Identify raw variables (those not starting with "kpi_")
raw_cols = [col for col in df_financial.columns if not col.startswith("kpi_")]

raw_cols

['company_name',
 'siren',
 'siret',
 'caConsolide',
 'caGroupe',
 'resultatExploitation',
 'dateConsolide']

In [113]:
# Identify kpi variables
kpi_cols = [col for col in df_financial.columns if col.startswith("kpi_")]
kpi_cols

['kpi_2025_capital_social',
 'kpi_2025_evolution_ca',
 'kpi_2023_ca_france',
 'kpi_2023_ca_bilan',
 'kpi_2023_resultat_exploitation',
 'kpi_2023_capital_social',
 'kpi_2023_resultat_bilan',
 'kpi_2022_ca_france',
 'kpi_2022_ca_bilan',
 'kpi_2022_resultat_exploitation',
 'kpi_2022_capital_social',
 'kpi_2022_resultat_bilan',
 'kpi_2021_ca_france',
 'kpi_2021_ca_bilan',
 'kpi_2021_resultat_exploitation',
 'kpi_2021_capital_social',
 'kpi_2021_resultat_bilan',
 'kpi_2020_ca_france',
 'kpi_2020_ca_bilan',
 'kpi_2020_resultat_exploitation',
 'kpi_2020_capital_social',
 'kpi_2020_resultat_bilan',
 'kpi_2019_ca_france',
 'kpi_2019_ca_bilan',
 'kpi_2019_resultat_exploitation',
 'kpi_2019_capital_social',
 'kpi_2019_resultat_bilan',
 'kpi_2018_ca_bilan',
 'kpi_2018_resultat_exploitation',
 'kpi_2018_capital_social',
 'kpi_2018_resultat_bilan',
 'kpi_2017_ca_france',
 'kpi_2017_ca_bilan',
 'kpi_2017_resultat_exploitation',
 'kpi_2017_capital_social',
 'kpi_2017_resultat_bilan',
 'kpi_2016_ca_bil

In [114]:
# Extract the KPI base name (ex: 'ca_bilan', 'resultat_exploitation', etc.)
def strip_year(col):
    return re.sub(r'kpi_\d{4}_', '', col)  # removes "kpi_2024_"

# Compare each raw column against matching KPI columns
results = {}

for raw in raw_cols:
    # find KPI columns that correspond to the raw variable
    raw_normalised = raw.lower().replace("consolide", "consolide").replace("exploitation", "exploitation")

    related_kpis = [k for k in kpi_cols if strip_year(k).lower() == raw_normalised]

    if not related_kpis:
        results[raw] = "⚠️ No corresponding KPI-year column found"
        continue

    # compare values
    match_info = {}
    for kcol in related_kpis:
        same = (df_financial[raw] == df_financial[kcol]).sum()
        total = df_financial.shape[0]
        match_ratio = same / total
        match_info[kcol] = round(match_ratio, 4)

    results[raw] = match_info

results


{'company_name': '⚠️ No corresponding KPI-year column found',
 'siren': '⚠️ No corresponding KPI-year column found',
 'siret': '⚠️ No corresponding KPI-year column found',
 'caConsolide': '⚠️ No corresponding KPI-year column found',
 'caGroupe': '⚠️ No corresponding KPI-year column found',
 'resultatExploitation': '⚠️ No corresponding KPI-year column found',
 'dateConsolide': '⚠️ No corresponding KPI-year column found'}

In [115]:
# Select only financial columns with at least 150 non-null values
min_non_null = 150

financial_selected_cols = (
    df_financial
        .drop(columns=['company_name', 'siren', 'siret'])      # keep only metrics
        .count()
        .loc[lambda x: x >= min_non_null]
        .index.tolist()
)

print("Selected FINANCIAL columns:", financial_selected_cols)

df_financial_feat = df_financial[['siren'] + financial_selected_cols].copy()


Selected FINANCIAL columns: ['caConsolide', 'caGroupe', 'resultatExploitation', 'dateConsolide', 'kpi_2022_ca_bilan', 'kpi_2022_capital_social', 'kpi_2022_resultat_bilan', 'kpi_2021_ca_bilan', 'kpi_2021_resultat_exploitation', 'kpi_2021_capital_social', 'kpi_2021_resultat_bilan', 'kpi_2020_ca_france', 'kpi_2020_ca_bilan', 'kpi_2020_resultat_exploitation', 'kpi_2020_capital_social', 'kpi_2020_resultat_bilan', 'kpi_2019_ca_france', 'kpi_2019_ca_bilan', 'kpi_2019_resultat_exploitation', 'kpi_2019_capital_social', 'kpi_2019_resultat_bilan', 'kpi_2018_ca_bilan', 'kpi_2018_resultat_exploitation', 'kpi_2018_capital_social', 'kpi_2018_resultat_bilan', 'kpi_2017_ca_france', 'kpi_2017_ca_bilan', 'kpi_2017_resultat_exploitation', 'kpi_2017_capital_social', 'kpi_2017_resultat_bilan', 'kpi_2016_ca_bilan', 'kpi_2016_resultat_bilan', 'kpi_2015_ca_bilan', 'kpi_2015_resultat_bilan', 'kpi_2014_ca_bilan', 'kpi_2014_resultat_bilan', 'kpi_2018_ca_france', 'kpi_2016_ca_france', 'kpi_2016_resultat_exploitati

In [116]:
df_financial_feat.head()

,siren,caConsolide,caGroupe,resultatExploitation,dateConsolide,kpi_2022_ca_bilan,kpi_2022_capital_social,kpi_2022_resultat_bilan,kpi_2021_ca_bilan,kpi_2021_resultat_exploitation,...,kpi_2015_resultat_bilan,kpi_2014_ca_bilan,kpi_2014_resultat_bilan,kpi_2018_ca_france,kpi_2016_ca_france,kpi_2016_resultat_exploitation,kpi_2016_capital_social,kpi_2013_ca_bilan,kpi_2013_resultat_bilan,kpi_2024_capital_social
0,000132066,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,015751530,0.0,0.0,76546.0,0.0,6247357.0,120000.0,182001.0,5296275.0,-62109.0,...,181450.0,6653070.0,388230.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,016450298,0.0,0.0,670860.0,0.0,8678668.0,257600.0,453579.0,7588920.0,429064.0,...,NaN,NaN,NaN,6954431.0,NaN,NaN,NaN,NaN,NaN,NaN
3,046580031,0.0,0.0,140333.0,0.0,NaN,350000.0,354077.0,NaN,NaN,...,140570.0,4068800.0,168950.0,NaN,3215827.0,-317.0,350000.0,3760470.0,199020.0,NaN
4,057504649,0.0,0.0,473736.0,0.0,NaN,1000000.0,180831.0,NaN,NaN,...,298258.0,6125829.0,115160.0,NaN,6489800.0,585559.0,1000000.0,5639506.0,90561.0,NaN


In [117]:
df_workforce.columns

Index(['company_name', 'siren', 'siret', 'effectif', 'effectifConsolide',
       'effectifEstime', 'effectifGroupe'],
      dtype='object')

In [118]:
df_workforce.isnull().mean().sort_values(ascending=False)

effectifConsolide    0.205333
effectifEstime       0.205333
effectifGroupe       0.205333
siret                0.064000
effectif             0.061333
company_name         0.000000
siren                0.000000
dtype: float64

In [119]:
df_workforce.shape

(375, 7)

In [120]:
# WORKFORCE FEATURES
df_workforce_feat = df_workforce[['siren', 'effectif', 'effectifConsolide']].copy()

# Convert to numeric (if needed)
# workforce_feat['effectif'] = pd.to_numeric(workforce_feat['effectif'], errors='coerce')


In [121]:
df_structure.columns

Index(['company_name', 'siren', 'siret', 'nbFilialesDirectes',
       'nbEtabSecondaire', 'nbMarques', 'hasGroupOwner', 'appartient_groupe',
       'nombre_etablissements_secondaires_inactifs'],
      dtype='object')

In [122]:
df_structure.shape

(375, 9)

In [123]:
# STRUCTURE FEATURES

structure_selected = [
    'siren',
    'nbFilialesDirectes',
    'nbEtabSecondaire',
    'nbMarques',
    'hasGroupOwner',   # or 'appartient_groupe'
    'nombre_etablissements_secondaires_inactifs'
]

df_structure_feat = df_structure[structure_selected].copy()
df_structure_feat.head()


,siren,nbFilialesDirectes,nbEtabSecondaire,nbMarques,hasGroupOwner,nombre_etablissements_secondaires_inactifs
0,000132066,0.0,0.0,0.0,False,0
1,015751530,6.0,4.0,14.0,False,3
2,016450298,1.0,0.0,4.0,False,2
3,046580031,0.0,0.0,3.0,False,0
4,057504649,0.0,0.0,0.0,False,0


In [124]:
df_flags.columns

Index(['company_name', 'siren', 'siret', 'startup', 'radiee', 'entreprise_b2b',
       'entreprise_b2c', 'fintech', 'cac40', 'entreprise_familiale',
       'entreprise_familiale_ter', 'filtre_levee_fond', 'flag_type_entreprise',
       'hasMarques', 'hasESV1Contacts'],
      dtype='object')

In [125]:
df_flags.shape

(375, 15)

In [126]:
df_flags.isnull().mean().sort_values(ascending=False)

siret                       0.064000
entreprise_familiale_ter    0.061333
company_name                0.000000
siren                       0.000000
startup                     0.000000
radiee                      0.000000
entreprise_b2b              0.000000
entreprise_b2c              0.000000
fintech                     0.000000
cac40                       0.000000
entreprise_familiale        0.000000
filtre_levee_fond           0.000000
flag_type_entreprise        0.000000
hasMarques                  0.000000
hasESV1Contacts             0.000000
dtype: float64

In [127]:
# FLAG FEATURES
df_flags_feat = df_flags[['siren','filtre_levee_fond', 'hasMarques']].copy()

df_flags_feat.head()

,siren,filtre_levee_fond,hasMarques
0,000132066,False,False
1,015751530,False,True
2,016450298,False,True
3,046580031,False,True
4,057504649,False,False


In [128]:
df_signals.columns

Index(['company_name', 'siren', 'siret', 'continent', 'country', 'departement',
       'publishedAt', 'isMain', 'type', 'createdAt', 'companies_count',
       'sirets_count'],
      dtype='object')

In [129]:
df_signals.shape

(2133, 12)

In [130]:
df_signals.groupby('siren').sum().shape

(266, 11)

In [131]:
# Prepare signals
df_signals_feat = df_signals[['siren', 'type', 'publishedAt']].copy()
df_signals_feat.head()

,siren,type,publishedAt
0,015751530,"{'code': 'K1', 'id': 32, 'label': 'Investissem...",2021-09-30T00:00:00+02:00
1,015751530,"{'code': 'L', 'id': 12, 'label': 'Levée de fon...",2020-09-08T00:00:00+02:00
2,015751530,"{'code': 'F', 'id': 6, 'label': ""Développement...",2016-09-21T00:00:00+02:00
3,015751530,"{'code': 'F', 'id': 6, 'label': ""Développement...",2018-04-06T00:00:00+02:00
4,015751530,"{'code': 'E', 'id': 5, 'label': 'Créations & o...",2018-04-06T00:00:00+02:00


In [132]:
df_signals_feat['type'].unique()


array(["{'code': 'K1', 'id': 32, 'label': 'Investissements'}",
       "{'code': 'L', 'id': 12, 'label': 'Levée de fonds, financements & modifs. capital'}",
       '{\'code\': \'F\', \'id\': 6, \'label\': "Développement de l\'activité"}',
       "{'code': 'E', 'id': 5, 'label': 'Créations & ouvertures'}",
       "{'code': 'H', 'id': 8, 'label': 'Activité internationale (tertiaire)'}",
       "{'code': 'X', 'id': 25, 'label': 'Actualité entreprise'}",
       "{'code': 'U', 'id': 21, 'label': 'Nomination'}",
       "{'code': 'Hbis', 'id': 24, 'label': 'Activité internationale (industrie)'}",
       "{'code': 'B', 'id': 2, 'label': 'Construction'}",
       "{'code': 'S', 'id': 19, 'label': 'Lancement'}",
       "{'code': 'N', 'id': 14, 'label': 'Recrutement'}",
       "{'code': 'N', 'label_short': 'Recrutement', 'id': 14, 'label': 'Recrutement'}",
       "{'code': 'Z2', 'id': 30, 'label': 'Restructuration, Réorganisation'}",
       "{'code': 'Z1', 'id': 29, 'label': 'Engagement vert'}",
  

In [133]:
import ast

# Convert the string dict into a real dict
df_signals_feat['type'] = df_signals_feat['type'].apply(ast.literal_eval)

# Extract the label
df_signals_feat['signal_label'] = df_signals_feat['type'].apply(lambda x: x['label'])


In [134]:
# Extract the code
df_signals_feat['signal_code'] = df_signals_feat['type'].apply(lambda x: x['code'])


In [135]:
df_signals_feat.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2133 entries, 0 to 2132
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   siren         2133 non-null   object
 1   type          2133 non-null   object
 2   publishedAt   2133 non-null   object
 3   signal_label  2133 non-null   object
 4   signal_code   2133 non-null   object
dtypes: object(5)
memory usage: 83.4+ KB


In [136]:
df_signals_feat['signal_label'].value_counts()


signal_label
Construction                                      335
Investissements                                   295
Recrutement                                       233
Levée de fonds, financements & modifs. capital    230
Développement de l'activité                       210
Activité internationale (industrie)               150
Vente & Cession                                   126
Lancement                                          92
Créations & ouvertures                             63
Nomination                                         48
Engagement vert                                    45
Activité internationale (tertiaire)                42
Déménagement                                       38
Décroissance                                       34
Foncier & bâti                                     33
Actualité entreprise                               33
Restructuration, Réorganisation                    30
Licenciement & chômage                             26
Politique RH & 

In [137]:
df_signals_feat['signal_code'].value_counts()

signal_code
B       335
K1      295
N       233
L       230
F       214
Hbis    150
P       126
S        92
E        70
U        48
Z1       45
H        42
G        38
R        34
X        33
W        33
Z2       30
M        26
D        17
Z3       15
O        11
I        10
A         3
Y         2
Z         1
Name: count, dtype: int64

In [138]:
df_signals_feat[['signal_code', 'signal_label']].drop_duplicates().sort_values('signal_code')


,signal_code,signal_label
568,A,Aménagement & urbanisme
11,B,Construction
218,D,Litige
155,D,Litiges
314,E,Création & ouverture
4,E,Créations & ouvertures
2,F,Développement de l'activité
632,F,Croissance
93,G,Déménagement
5,H,Activité internationale (tertiaire)


In [140]:
df_signals_feat['publishedAt'] = pd.to_datetime(df_signals_feat['publishedAt'], utc=True)


In [143]:
cutoff = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=2)
df_recent_signals = df_signals_feat[df_signals_feat['publishedAt'] >= cutoff]

In [144]:
df_recent_signals.head()

,siren,type,publishedAt,signal_label,signal_code
17,015751530,"{'code': 'N', 'label_short': 'Recrutement', 'i...",2025-03-02 23:15:21+00:00,Recrutement,N
25,057504649,"{'code': 'B', 'id': 2, 'label': 'Construction'}",2024-06-15 22:00:00+00:00,Construction,B
30,096780838,"{'code': 'Z2', 'id': 30, 'label': 'Restructura...",2024-03-10 23:00:00+00:00,"Restructuration, Réorganisation",Z2
32,096780838,"{'code': 'Z1', 'id': 29, 'label': 'Engagement ...",2024-03-10 23:00:00+00:00,Engagement vert,Z1
35,099378564,"{'code': 'F', 'label_short': 'Croissance', 'id...",2025-09-24 22:15:40+00:00,Développement de l'activité,F


In [145]:
df_recent_signals.shape

(447, 5)

In [ ]:
positive_codes = ['B', 'K1', 'F', 'E', 'S']
negative_codes = ['M', 'O', 'I']

def classify_signal(code):
    if code in positive_codes:
        return 'positive'
    elif code in negative_codes:
        return 'negative'
    else:
        return 'neutral'

df_recent_signals['signal_valence'] = df_recent_signals['signal_code'].apply(classify_signal)


In [93]:
df_articles.columns

Index(['company_name', 'siren', 'siret', 'title', 'publishedAt', 'author',
       'signalsStatus', 'signalsType', 'country', 'sectors', 'cities',
       'sources', 'all_companies_count'],
      dtype='object')

In [94]:
df_articles.shape

(1180, 13)

In [95]:
df_articles.groupby('siren').sum().shape

(266, 12)

## The information provided by df_articles is already treated and duplicated in df_signals, so this dataset won't be used.

In [96]:
# Load KPI data
df_kpi = pd.read_csv(DATA_DIR / '07_kpi_data.csv', dtype={'siren': str, 'siret': str})

print(f"Dataset shape: {df_kpi.shape}")
print(f"\nColumns: {list(df_kpi.columns)}")
display(df_kpi.head(5))

Dataset shape: (3779, 28)

Columns: ['company_name', 'siren', 'siret', 'year', 'fonds_propres', 'ca_france', 'date_cloture_exercice', 'duree_exercice', 'salaires_traitements', 'charges_financieres', 'impots_taxes', 'ca_bilan', 'resultat_exploitation', 'dotations_amortissements', 'capital_social', 'code_confidentialite', 'resultat_bilan', 'annee', 'effectif', 'effectif_sous_traitance', 'filiales_participations', 'evolution_ca', 'subventions_investissements', 'ca_export', 'evolution_effectif', 'participation_bilan', 'ca_consolide', 'resultat_net_consolide']


,company_name,siren,siret,year,fonds_propres,ca_france,date_cloture_exercice,duree_exercice,salaires_traitements,charges_financieres,...,effectif,effectif_sous_traitance,filiales_participations,evolution_ca,subventions_investissements,ca_export,evolution_effectif,participation_bilan,ca_consolide,resultat_net_consolide
0,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2023,2192166.0,6729652.0,2023-01-31,12.0,1394492.0,80993.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2022,1614077.0,6247357.0,2022-01-31,12.0,1327711.0,81469.0,...,35.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2021,1497114.0,5296275.0,2021-01-31,12.0,1318083.0,66111.0,...,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2020,1577275.0,5710890.0,2020-01-31,12.0,1380952.0,70953.0,...,45.0,18930.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2019,1348804.0,5221375.0,2019-01-31,12.0,1230571.0,88389.0,...,43.0,15835.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
df_kpi.columns.tolist()

['company_name',
 'siren',
 'siret',
 'year',
 'fonds_propres',
 'ca_france',
 'date_cloture_exercice',
 'duree_exercice',
 'salaires_traitements',
 'charges_financieres',
 'impots_taxes',
 'ca_bilan',
 'resultat_exploitation',
 'dotations_amortissements',
 'capital_social',
 'code_confidentialite',
 'resultat_bilan',
 'annee',
 'effectif',
 'effectif_sous_traitance',
 'filiales_participations',
 'evolution_ca',
 'subventions_investissements',
 'ca_export',
 'evolution_effectif',
 'participation_bilan',
 'ca_consolide',
 'resultat_net_consolide']

In [98]:
feature_cols = [
    'fonds_propres',
    'resultat_exploitation',
    #'effectif'
    #'recency'   # giving weight to fresh companies
]

# Keeping only modelling columns + identifiers
cols_to_keep = ['company_name', 'siren'] + feature_cols
df_kpi_feat = df_kpi[cols_to_keep].copy()

df_kpi_feat.head()

,company_name,siren,fonds_propres,resultat_exploitation
0,PAIN D'EPICES MULOT ET PETITJEAN,015751530,2192166.0,76546.0
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1614077.0,65908.0
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1497114.0,-62109.0
3,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1577275.0,93684.0
4,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1348804.0,-247638.0


In [99]:
df_kpi_feat.shape

(3779, 4)

In [ ]:
### Merge everything
df_merged = (
    df_basic_feat
    .merge(df_workforce_feat, on='siren', how='left')
    .merge(df_structure_feat, on='siren', how='left')
    .merge(df_flags_feat, on='siren', how='left')
)


In [ ]:
df_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2242 entries, 0 to 2241
Data columns (total 14 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   company_name                                2242 non-null   object 
 1   siren                                       2242 non-null   object 
 2   siret                                       2218 non-null   float64
 3   departement                                 2242 non-null   object 
 4   effectif                                    2219 non-null   float64
 5   nbFilialesDirectes                          2152 non-null   float64
 6   nbEtabSecondaire                            2152 non-null   float64
 7   nbMarques                                   2152 non-null   float64
 8   hasGroupOwner                               2242 non-null   bool   
 9   nombre_etablissements_secondaires_inactifs  2242 non-null   int64  
 10  filtre_levee

In [74]:
df_merged.head()

,company_name,siren,siret,departement,effectif,nbFilialesDirectes,nbEtabSecondaire,nbMarques,hasGroupOwner,nombre_etablissements_secondaires_inactifs,filtre_levee_fond,hasMarques,type,signal_createdAt
0,PROCONI,000132066,NaN,00,0.0,0.0,0.0,0.0,False,0,False,False,NaN,NaN
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'K1', 'id': 32, 'label': 'Investissem...",2020-09-07T15:14:38+02:00
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'L', 'id': 12, 'label': 'Levée de fon...",2020-09-07T15:14:12+02:00
3,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'F', 'id': 6, 'label': ""Développement...",2016-09-20T10:45:13+02:00
4,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'F', 'id': 6, 'label': ""Développement...",2018-04-05T11:16:18+02:00


In [76]:
df_merged.isnull().mean().sort_values(ascending=False)

type                                          0.048617
signal_createdAt                              0.048617
nbFilialesDirectes                            0.040143
nbEtabSecondaire                              0.040143
nbMarques                                     0.040143
siret                                         0.010705
effectif                                      0.010259
company_name                                  0.000000
siren                                         0.000000
departement                                   0.000000
hasGroupOwner                                 0.000000
nombre_etablissements_secondaires_inactifs    0.000000
filtre_levee_fond                             0.000000
hasMarques                                    0.000000
dtype: float64

In [79]:
df_merged.describe().T

,count,mean,std,min,25%,50%,75%,max
siret,2218.0,4.752240e+13,1.595535e+13,1.575153e+12,3.507062e+13,4.443239e+13,5.388667e+13,9.504515e+13
effectif,2219.0,6.282965e+01,6.880793e+01,0.000000e+00,3.500000e+01,3.500000e+01,7.500000e+01,3.750000e+02
nbFilialesDirectes,2152.0,5.631970e-01,1.733447e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.000000e+00
nbEtabSecondaire,2152.0,1.940056e+00,5.740526e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,9.300000e+01
nbMarques,2152.0,1.056924e+01,1.924768e+01,0.000000e+00,0.000000e+00,2.000000e+00,1.100000e+01,8.100000e+01
nombre_etablissements_secondaires_inactifs,2242.0,2.174398e+00,4.374218e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00,6.700000e+01


In [80]:
df_merged.shape

(2242, 14)

In [82]:
df_merged.dtypes

company_name                                   object
siren                                          object
siret                                         float64
departement                                    object
effectif                                      float64
nbFilialesDirectes                            float64
nbEtabSecondaire                              float64
nbMarques                                     float64
hasGroupOwner                                    bool
nombre_etablissements_secondaires_inactifs      int64
filtre_levee_fond                                bool
hasMarques                                       bool
type                                           object
signal_createdAt                               object
dtype: object